# Pseudobulk model building — PLIER

**Environment:** `clamp-analyses`

Runs PLIER with GO Biological Process prior on every pseudobulk dataset. Preprocesses raw counts from `bulk_expr.csv`. Reads `k.csv` from the CLAMP notebook output to use the same rank estimate. Outputs written to `output/01_model_building/05_pseudobulk/<dataset>/PLIER/`.

## Libraries

In [1]:
library(data.table)
library(dplyr)
library(rsvd)
library(Matrix)
library(here)
library(CLAMP)
library(PLIER)
library(PCAtools)

set.seed(123)


Attaching package: ‘dplyr’




The following objects are masked from ‘package:data.table’:

    between, first, last




The following objects are masked from ‘package:stats’:

    filter, lag




The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



...

The following object is masked from ‘package:CLAMP’:

    num.pc




Loading required package: ggplot2



Loading required package: ggrepel




Attaching package: ‘PCAtools’




The following objects are masked from ‘package:stats’:

    biplot, screeplot




## Configuration

In [2]:
DATASET  = "PBMC_Perez2022"
OUT_ROOT = "output/01_model_building/05_pseudobulk"
DATA_DIR = "data/pseudobulk"

In [3]:
# Parameters
DATASET = "Lung_Sikkema2023"


## Download BP pathway GMT (cached)

In [4]:
gmt_raw <- list(
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
)
for (lib in names(gmt_raw)) {
  names(gmt_raw[[lib]]) <- paste0(lib, "_", names(gmt_raw[[lib]]))
}
pathMat_raw <- gmtListToSparseMat(gmt_raw)
cat("Pathway matrix:", nrow(pathMat_raw), "genes x", ncol(pathMat_raw), "pathways\n")

Auto-detected name: GO_Biological_Process_2025



Using cached file for GO_Biological_Process_2025



Pathway matrix: 14674 genes x 5343 pathways


## Build PLIER model for each dataset

In [5]:
message("========== ", DATASET, " ==========")
out_dir <- file.path(here(), OUT_ROOT, DATASET)

# Load preprocessed data
norm_dt    <- fread(file.path(here(), OUT_ROOT, DATASET, "norm.csv"))
norm_genes <- norm_dt[[1]]
norm       <- as.matrix(norm_dt[, -1, with = FALSE])
storage.mode(norm) <- "numeric"
rownames(norm) <- norm_genes
samples <- colnames(norm)
cat(DATASET, "norm:", nrow(norm), "genes x", ncol(norm), "samples\n")

# Load k
k <- as.integer(read.csv(file.path(here(), OUT_ROOT, DATASET, "k.csv"))$k[1])
message("  k = ", k)

# SVD (for PLIER warm start)
g_fb       <- nrow(norm)
samples_fb <- ncol(norm)
SVD_K      <- floor((min(g_fb, samples_fb) - 1) / 4)
svdres <- rsvd(norm, k = SVD_K)

# Match BP prior to dataset genes
matched  <- getMatchedPathwayMat(pathMat_raw, norm_genes)
chatObj  <- getChat(matched)
cat("  Prior matched:", nrow(matched), "genes x", ncol(matched), "pathways\n")

# PLIER
message("  Running PLIER ...")
plier_res <- PLIER::PLIER(
  norm,
  as.matrix(matched),
  svdres     = svdres,
  Chat       = as.matrix(chatObj),
  doCrossval = FALSE,
  k          = k
)

colnames(plier_res$Z) <- paste0("LV", seq_len(ncol(plier_res$Z)))
plier_res$summary <- plier_res$summary %>%
  dplyr::rename(LV = `LV index`) %>%
  dplyr::mutate(LV = paste0("LV", LV))
colnames(plier_res$B) <- samples

model_dir <- file.path(out_dir, "PLIER")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
write.csv(plier_res$B, file.path(model_dir, "B.csv"))
write.csv(plier_res$Z, file.path(model_dir, "Z.csv"))
write.csv(plier_res$summary, file.path(model_dir, "summary.csv"), row.names = FALSE)
saveRDS(plier_res, file.path(model_dir, "PLIER.rds"))
message("  PLIER saved -> ", model_dir)

========== Lung_Sikkema2023 ==========



Lung_Sikkema2023 norm: 17145 genes x 100 samples


  k = 18



There are 11839 genes in the intersection between data and prior



Removing 2090 pathways



...

New L3 is 0.000261258557301668



converged at  iteration 137



Not using cross-validation. AUCs and p-values may be over-optimistic



There are 6  LVs with AUC>0.70



  PLIER saved -> /home/msubirana/Documents/pivlab/clamp-analyses/output/01_model_building/05_pseudobulk/Lung_Sikkema2023/PLIER

